In [1]:
!git clone https://github.com/TsinghuaC3I/MedXpertQA.git


Cloning into 'MedXpertQA'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 75 (delta 18), reused 69 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 7.17 MiB | 32.94 MiB/s, done.
Resolving deltas: 100% (18/18), done.


In [1]:
import os

# Check current working directory
print("Current directory:", os.getcwd())

# List all folders/files to confirm path
for root, dirs, files in os.walk(".", topdown=True):
    for name in files:
        if name.endswith(".jsonl"):
            print(os.path.join(root, name))

import pandas as pd

# Ensure full content is shown
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)


Current directory: /data/healthy-ml/scratch/yuexing/NeuRIPS25/MedXpertQA
./MedXpertQA/eval/data/mmlu_medical/input/mmlu_medical_input.jsonl
./MedXpertQA/eval/data/medmcqa/input/medmcqa_input.jsonl
./MedXpertQA/eval/data/medqa/input/medqa_input.jsonl
./MedXpertQA/eval/data/medxpertqa/input/medxpertqa_text_input.jsonl
./MedXpertQA/eval/data/medxpertqa/input/medxpertqa_mm_input.jsonl
./MedXpertQA/eval/data/medxpertqa_sampled/input/medxpertqa_sampled_text_input.jsonl
./MedXpertQA/eval/data/medxpertqa_sampled/input/medxpertqa_sampled_mm_input.jsonl


In [1]:
import json

text_path = "./MedXpertQA/eval/data/medxpertqa/input/medxpertqa_text_input.jsonl"
mm_path = "./MedXpertQA/eval/data/medxpertqa/input/medxpertqa_mm_input.jsonl"


def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

# Load datasets
text_data = load_jsonl(text_path)
mm_data = load_jsonl(mm_path)

# Print dataset lengths
print("Text-only dataset length:", len(text_data))
print("Multimodal dataset length:", len(mm_data))


Text-only dataset length: 2450
Multimodal dataset length: 2000


In [2]:
import pandas as pd

# Convert the list of JSON objects to a DataFrame
text_df = pd.DataFrame(text_data)

# # Show the first few rows
print(text_df.head())

# # List all column names
print("\nColumns in text_df:")
# Show full content of the first row with key-value structure

# View first row fully
import pprint

# Convert the first row to a dictionary and pretty-print it
pprint.pprint(text_df.iloc[0].to_dict(), width=200)



       id                                           question  \
0  Text-0  Which patient scenario represents the most app...   
1  Text-1  A 55-year-old postmenopausal woman reports exp...   
2  Text-2  A newborn develops tachypnea and retractions s...   
3  Text-3  A 34-year-old woman describes intermittent mil...   
4  Text-4  Over the past year, a pediatric cardiac surgeo...   

                                             options label   medical_task  \
0  [{'letter': 'A', 'content': '70-year-old male ...   [E]  Basic Science   
1  [{'letter': 'A', 'content': 'Sacrotuberous lig...   [E]      Diagnosis   
2  [{'letter': 'A', 'content': 'Respiratory suppo...   [C]      Diagnosis   
3  [{'letter': 'A', 'content': 'Begin octreotide ...   [I]      Treatment   
4  [{'letter': 'A', 'content': 'A 5-year-old boy ...   [J]      Diagnosis   

      body_system question_type  
0        Skeletal     Reasoning  
1        Muscular     Reasoning  
2     Respiratory     Reasoning  
3       Endocrin

In [3]:
# List unique values in the 'medical_task' column
unique_tasks = text_df["medical_task"].unique()
print("Unique medical_task types:", unique_tasks)

# Optionally, count how often each occurs
task_counts = text_df["medical_task"].value_counts()
print("\nFrequency of each medical_task type:\n", task_counts)

# List unique values in the 'question_type' column
unique_types = text_df["question_type"].unique()
print("Unique question_type values:", unique_types)

# Count frequency of each type
type_counts = text_df["question_type"].value_counts()
print("\nFrequency of each question_type value:\n", type_counts)


Unique medical_task types: ['Basic Science' 'Diagnosis' 'Treatment']

Frequency of each medical_task type:
 medical_task
Diagnosis        1050
Treatment         746
Basic Science     654
Name: count, dtype: int64
Unique question_type values: ['Reasoning' 'Understanding']

Frequency of each question_type value:
 question_type
Reasoning        1861
Understanding     589
Name: count, dtype: int64


In [4]:
reasoning_df = text_df[text_df["question_type"] == "Reasoning"]
print(reasoning_df.shape)           # Shows the number of rows and columns
print(reasoning_df.head(1))         # Shows the first row
print(reasoning_df["question_type"].unique())  # Sanity check — should return ['Reasoning']


(1861, 7)
       id                                           question  \
0  Text-0  Which patient scenario represents the most app...   

                                             options label   medical_task  \
0  [{'letter': 'A', 'content': '70-year-old male ...   [E]  Basic Science   

  body_system question_type  
0    Skeletal     Reasoning  
['Reasoning']


## Direct Prediction for "Reasoning"

In [17]:
import openai

def generate_reasoning_prediction(row):
    """
    Uses GPT-4o to predict the correct answer from a medical multiple-choice question.
    Handles any number of answer options dynamically.
    """
    # Construct the formatted list of choices
    formatted_choices = "\n".join(
        f"{opt['letter']}. {opt['content']}" for opt in row['options']
    )

    prompt = f"""
The following is a medical multiple-choice question. It includes a clinical vignette followed by answer options labeled A, B, C, etc.

Question:
{row['question']}

Choices:
{formatted_choices}

Your task:
- Select the best answer.
- Respond strictly in the following format: [Letter]: [Answer Text] (e.g., B: Femoral artery murmur)
- Do not explain. Do not repeat the question.
"""

    try:
        client = openai.OpenAI()

        response = client.chat.completions.create(
            model="o3-mini", #gpt-4o
            messages=[{"role": "user", "content": prompt}]
        )

        return response.choices[0].message.content.strip()

    except Exception as e:
        return f"Error: {e}"


In [18]:
from tqdm.notebook import tqdm  # Optional: for progress bar

test_df = reasoning_df.head(1).copy()
test_df["gpto3_prediction"] = None  # Initialize the prediction column

for idx in tqdm(test_df.index):
    row = test_df.loc[idx]
    prediction = generate_reasoning_prediction(row)
    test_df.at[idx, "gpto3_prediction"] = prediction

    # Show progress in real-time
    print(f"\nRow {idx} | Prediction: {prediction} | Ground Truth: {row['label']}")

    # Save progress every 50 iterations
    if idx % 50 == 0:
        test_df.to_csv("reasoning_predictions_partial.csv", index=False)


  0%|          | 0/1 [00:00<?, ?it/s]


Row 0 | Prediction: C: 63-year-old female with glenoid retroversion of 22-degrees and mild posterior wear undergoing shoulder arthroplasty | Ground Truth: ['E']


In [62]:
# Extract the predicted letter from GPT output (e.g., 'A' from 'A: ...')
test_df["gpt_letter"] = test_df["gpto3_prediction"].astype(str).str.strip().str[0]

# Extract the ground truth letter from the label list
test_df["answer_letter"] = test_df["label"].apply(lambda x: str(x[0]).strip().upper() if isinstance(x, list) and x else None)

# Compare predicted letter with ground truth
test_df["gpt_letter_match"] = test_df.apply(
    lambda row: "Correct" if row["gpt_letter"] == row["answer_letter"] else "Incorrect",
    axis=1
)

# Compute accuracy
correct_count = (test_df["gpt_letter_match"] == "Correct").sum()
total_count = test_df["gpt_letter_match"].notna().sum()
accuracy = correct_count / total_count if total_count > 0 else 0

# Print results
print(f"Letter-Based Correct Predictions: {correct_count}")
print(f"Total Predictions Compared: {total_count}")
print(f"Letter Match Accuracy: {accuracy:.2%}")


Letter-Based Correct Predictions: 689
Total Predictions Compared: 1861
Letter Match Accuracy: 37.02%


In [29]:
# Select relevant columns to save
columns_to_save = ["gpt_prediction", "label", "gpt_letter", "answer_letter", "gpt_letter_match"]

# Save to CSV
test_df[columns_to_save].to_csv("gpt_predictions_comparison.csv", index=False)

print("Saved predictions and comparison to 'gpt_predictions_comparison.csv'")


Saved predictions and comparison to 'gpt_predictions_comparison.csv'


## Merge into Actual_question

## For Centaur Lab Numbered List

In [5]:
import pandas as pd
import re

# Load your data
text_data = pd.read_csv("processed_textdata_with_actual_question.csv")

# Keywords that often precede lab-like sections
lab_section_headers = ['Laboratory results', 'Hematologic', 'Serum', 'Urine', 'CSF', 'Imaging', 'Vitals', 'Exam', 'Findings']

def is_lab_section(line):
    return any(keyword.lower() in line.lower() for keyword in lab_section_headers)

# Main function to process and number question text properly
def number_question_vignette(text):
    if pd.isna(text):
        return text, 0

    lines = text.splitlines()
    output_lines = []
    count = 0
    in_lab_block = False
    question_started = False

    for line in lines:
        stripped = line.strip()

        # Skip blank lines but preserve spacing
        if not stripped:
            output_lines.append("")
            continue

        # If the question sentence starts, stop numbering
        if not question_started and stripped.endswith('?'):
            question_started = True
            output_lines.append(stripped)
            continue

        # If we're in the question or option section, just copy without numbering
        if question_started:
            output_lines.append(stripped)
            continue

        # Lab header — set flag and output as-is
        if is_lab_section(stripped):
            output_lines.append(stripped)
            in_lab_block = True
            continue

        # Number lab-like bullet if in lab section
        if stripped.startswith('-') and in_lab_block:
            count += 1
            converted = re.sub(r'^-\s*', f"{count}. ", stripped)
            output_lines.append(converted)
            continue

        # Number regular vignette lines
        count += 1
        output_lines.append(f"{count}. {stripped}")
        in_lab_block = False  # reset lab flag on regular line

    return "\n".join(output_lines), count

# Apply to the "question" column only
text_data[['numbered_question', 'number_of_sentences']] = text_data['question'].apply(
    lambda x: pd.Series(number_question_vignette(x))
)

# Preview
pd.set_option('display.max_colwidth', None)
print(text_data[['numbered_question', 'number_of_sentences']].head())
pd.reset_option('display.max_colwidth')

# Save to file
text_data.to_csv("processed_textdata_with_numbered_bullets.csv", index=False)

In [6]:
# Save the updated DataFrame to a CSV file
text_data.to_csv("processed_textdata_with_actual_question.csv", index=False)

print("File saved as 'processed_textdata_with_actual_question.csv'.")


File saved as 'processed_textdata_with_actual_question.csv'.


In [31]:
import pandas as pd
import re

# Load your data
text_data = pd.read_csv("processed_textdata_with_actual_question.csv")

# Keywords that trigger lab-like sections
lab_section_headers = ['Laboratory results', 'Hematologic', 'Serum', 'Urine', 'CSF', 'Imaging', 'CT', 'MRI', 'Vitals']

def is_lab_section(line):
    return any(keyword.lower() in line.lower() for keyword in lab_section_headers)

# Main function to process the question column
def process_question_with_lab_bullets(text):
    if pd.isna(text):
        return text, 0

    lines = text.splitlines()
    output_lines = []
    count = 0
    in_lab_block = False

    for line in lines:
        stripped = line.strip()
        if not stripped:
            output_lines.append("")  # keep spacing
            continue

        # Detect section trigger
        if is_lab_section(stripped):
            output_lines.append(stripped)
            in_lab_block = True
            continue

        # If it's a bullet and in a lab block, convert to numbered
        if stripped.startswith("-") and in_lab_block:
            count += 1
            converted = re.sub(r'^-\s*', f"{count}. ", stripped)
            output_lines.append(converted)
            continue

        # If it's a normal sentence, number it
        if not stripped.startswith("-"):
            count += 1
            output_lines.append(f"{count}. {stripped}")
            in_lab_block = False  # reset lab block on regular sentence
            continue

        # Else: fallback
        output_lines.append(stripped)

    return "\n".join(output_lines), count

# Apply to the 'question' column only
text_data[['numbered_question', 'number_of_sentences']] = text_data['question'].apply(
    lambda x: pd.Series(process_question_with_lab_bullets(x))
)

# Preview result
pd.set_option('display.max_colwidth', None)
print(text_data[['numbered_question', 'number_of_sentences']].head())
pd.reset_option('display.max_colwidth')

# Save to CSV
text_data.to_csv("processed_textdata_with_numbered_bullets.csv", index=False)


## For Bullet Points and Randomized

In [7]:
import pandas as pd
import re

# Function to process and format question text
def process_question_text(question_str):
    # Ensure input is string and remove extra whitespace
    question_str = str(question_str).strip()
    
    # Extract the question stem and options using a keyword like "Answer Choices:"
    split_parts = re.split(r'Answer Choices:\s*', question_str, maxsplit=1)
    
    if len(split_parts) != 2:
        return question_str  # Return as is if format is unexpected

    vignette, options_raw = split_parts
    # Split the vignette into sentences
    sentences = re.split(r'(?<=[.!?])\s+', vignette.strip())

    if len(sentences) > 1:
        # Use "-" instead of numbering
        formatted_vignette = "\n".join([f"- {s}" for s in sentences[:-1]])
        formatted_vignette += f"\n\n{sentences[-1]}"
    else:
        formatted_vignette = sentences[0]

    # Extract and format options
    option_lines = re.findall(r'\(([A-J])\)\s*([^()]+)(?=(?:\s+\([A-J]\))|\s*$)', options_raw)
    formatted_options = "\n\n" + "\n".join([f"{label}. {text.strip()}" for label, text in option_lines])

    return formatted_vignette + formatted_options

test_df = reasoning_df.head(1862).copy()

# Apply the function to the 'question' column and store in 'bullet_question'
test_df["bullet_question"] = test_df["question"].apply(process_question_text)

# Preview formatted results
pd.set_option('display.max_colwidth', None)
print(test_df[['question', 'bullet_question']].head(2))
pd.reset_option('display.max_colwidth')


In [8]:
import openai

def generate_direct_prediction(question_text):
    """
    Uses GPT-4o to predict the correct answer from a multiple-choice question embedded in a single string.
    Returns only the predicted answer in the format: 'B: Femoral artery murmur'
    """
    prompt = f"""
    The following is a medical multiple-choice question. It includes a clinical vignette followed by four answer options labeled A, B, C, and D.

    {question_text}

    Your task:
    - Select the best answer.
    - If the question refers to a figure or image, disregard it and focus solely on the text.
    - Respond strictly in the following format: [Letter]: [Answer Text] (e.g., B: Femoral artery murmur)
    - Do not explain. Do not repeat the question.
    """

    try:
        # Initialize OpenAI client
        client = openai.OpenAI(#Set the API key. See the how-to guide for further instructions
        )  # Set your API key
        # Generate response using GPT-4o
        response = client.chat.completions.create(
            model="o3-mini",
            messages=[{"role": "user", "content": prompt}]\
        )
        content = response.choices[0].message.content.strip()
        return content
    except Exception as e:
        return "Error"


In [ ]:
# Example: first 1034 rows for testing
test_df_subset = test_df.head(1862).copy()

# Apply GPT-4o direct prediction to the 'actual_question' column
# df_subset["gpto1_direct_prediction"] = df_subset["actual_question"].apply(generate_direct_prediction)
test_df_subset["gpto3_direct_prediction"] = test_df_subset["bullet_question"].apply(generate_direct_prediction)

# Show results
pd.set_option("display.max_colwidth", None)
display(text_data_subset[["bullet_question", "gpt4o_direct_prediction"]])
# display(df_subset[["actual_question", "gpto1_direct_prediction"]])

In [12]:
import pandas as pd
import time
from tqdm.notebook import tqdm

# Load your full DataFrame and subset the first 1862 rows
test_df_subset = test_df.head(1862).copy().reset_index(drop=True)

# Initialize the prediction column
test_df_subset["gpto3_direct_prediction"] = None

# If resuming from partial results
partial_path = "partial_gpto3_predictions.csv"

try:
    partial_df = pd.read_csv(partial_path)
    # Align the partial results with the subset by row index
    test_df_subset.loc[partial_df.index, "gpto3_direct_prediction"] = partial_df["gpto3_direct_prediction"]
    print(f"✅ Resumed from {partial_df['gpto3_direct_prediction'].notnull().sum()} previously completed rows.")
except FileNotFoundError:
    print("🆕 Starting fresh. No existing partial results found.")

# Run prediction with incremental saving
for i in tqdm(range(len(test_df_subset))):
    if pd.notnull(test_df_subset.iloc[i]["gpto3_direct_prediction"]):
        continue  # Skip already completed rows

    question = test_df_subset.iloc[i]["bullet_question"]
    try:
        prediction = generate_direct_prediction(question)
    except Exception as e:
        print(f"❌ Error at index {i}: {e}")
        prediction = "ERROR"

    test_df_subset.iat[i, test_df_subset.columns.get_loc("gpto3_direct_prediction")] = prediction

    # Save progress every 10 rows
    if i % 10 == 0:
        test_df_subset[["gpto3_direct_prediction"]].to_csv(partial_path, index=False)
        time.sleep(0.5)

# Final save
test_df_subset[["gpto3_direct_prediction"]].to_csv(partial_path, index=False)

# Display results
pd.set_option("display.max_colwidth", None)
display(test_df_subset[["bullet_question", "gpto3_direct_prediction"]])


✅ Resumed from 1531 previously completed rows.


  0%|          | 0/1861 [00:00<?, ?it/s]

,bullet_question,gpto3_direct_prediction
0,Which patient scenario represents the most appropriate indication for eccentric anterior glenoid reaming during shoulder surgery?\n\nA. 70-year-old male with glenoid retroversion of 18-degrees undergoing shoulder arthroplasty\nB. 70-year-old female with humeral anteversion of 13-degrees undergoing shoulder arthroplasty\nC. 63-year-old female with glenoid retroversion of 22-degrees and mild posterior wear undergoing shoulder arthroplasty\nD. 65-year-old female with glenoid retroversion of 25-degrees undergoing shoulder arthroplasty\nE. 65-year-old female with a glenoid retroversion of 13-degrees undergoing shoulder arthroplasty\nF. 68-year-old female with glenoid retroversion of 20-degrees undergoing reverse shoulder arthroplasty\nG. 72-year-old male with glenoid retroversion of 15-degrees undergoing shoulder arthroplasty\nH. 65-year-old female with glenoid retroversion of 30-degrees and severe posterior wear undergoing shoulder arthroplasty\nI. 58-year-old male with glenoid retroversion of 12-degrees undergoing shoulder arthroplasty\nJ. 55-year-old male with glenoid retroversion of 8-degrees undergoing total shoulder arthroplasty,E: 65-year-old female with a glenoid retroversion of 13-degrees undergoing shoulder arthroplasty
1,"- A 55-year-old postmenopausal woman reports experiencing sharp pain in the right groin for the past two weeks, which is alleviated by standing.\n- Her blood pressure is 140/92 mm Hg, and her heart rate is 88 bpm.\n- Cardiac auscultation reveals no murmurs or gallops, and abdominal, lung, and genitourinary examinations are unremarkable, with no palpable hernias.\n- On osteopathic evaluation, there is tenderness at L4 and L5 in the right paraspinal region.\n- The right sacral sulcus is shallow, and the right inferior lateral angle is posterior.\n- A seated flexion test is positive on the right.\n- Radiographic imaging of the hip and lumbar spine shows no acute or chronic abnormalities.\n\nWhich of the following structures is most likely implicated in the patient’s condition?\n\nA. Sacrotuberous ligament\nB. Quadratus lumborum muscle\nC. Piriformis muscle\nD. Posterior sacroiliac ligament\nE. Iliolumbar ligament\nF. Anterior sacroiliac ligament\nG. Psoas major muscle\nH. Sacrospinous ligament\nI. Gluteus medius muscle\nJ. Iliacus muscle",D: Posterior sacroiliac ligament
2,"- A newborn develops tachypnea and retractions shortly after birth, with a respiratory rate of 90/min.\n- Despite this, the infant shows no cyanosis, appears active, and does not look ill.\n- Blood gas analysis reveals no CO2 retention, and there are no other abnormal findings such as fever or rash.\n- The baby was delivered via cesarean section at 38 weeks following an uneventful pregnancy and is now under observation in intensive care.\n\nWhich of the following is the best management approach for this case?\n\nA. Respiratory support with mechanical ventilation is the first-line management\nB. The condition is benign and self-limiting, requiring no specific treatment\nC. This is hard to distinguish from pneumonia and sepsis so empirical antibiotics are given for 48 hours after birth until sepsis is ruled out\nD. Surfactant deficiency is the underlying cause, so surfactant therapy is required\nE. This is most likely transient tachypnea of the newborn and requires no antibiotics\nF. Tachypnea in this case is likely due to a congenital heart defect, so echocardiography is indicated\nG. ABG is needed to monitor and it is normal to have CO2 retention\nH. A chest X-ray must be done immediately to confirm the diagnosis of neonatal pneumonia\nI. Empirical antifungal therapy should be initiated until fungal sepsis is ruled out\nJ. Oxygen therapy is the mainstay of therapy with monitoring with SpO2",E: This is most likely transient tachypnea of the newborn and requires no antibiotics
3,"- A 34-year-old woman describes intermittent milky discharge from both breasts for the past 6 months.\n- She also stat

In [57]:
# Extract the predicted letter from GPT-4o's output (e.g., 'A' from 'A: ...')
test_df_subset["gpt_letter"] = test_df_subset["gpt4o_direct_prediction"].astype(str).str.strip().str[0]

# Extract the ground truth letter from the label list
test_df_subset["answer_letter"] = test_df_subset["label"].apply(
    lambda x: str(x[0]).strip().upper() if isinstance(x, list) and x else None
)

# Compare predicted letter with ground truth
test_df_subset["gpt_letter_match"] = test_df_subset.apply(
    lambda row: "Correct" if row["gpt_letter"] == row["answer_letter"] else "Incorrect",
    axis=1
)

# Compute accuracy
correct_count = (test_df_subset["gpt_letter_match"] == "Correct").sum()
total_count = test_df_subset["gpt_letter_match"].notna().sum()
accuracy = correct_count / total_count if total_count > 0 else 0

# Print results
print(f"Letter-Based Correct Predictions: {correct_count}")
print(f"Total Predictions Compared: {total_count}")
print(f"Letter Match Accuracy: {accuracy:.2%}")


Letter-Based Correct Predictions: 435
Total Predictions Compared: 1861
Letter Match Accuracy: 23.37%


In [ ]:
# Select relevant columns to save
columns_to_save = ["gpt_prediction", "label", "gpt_letter", "answer_letter", "gpt_letter_match"]

# Save to CSV
test_df_subsetp[columns_to_save].to_csv("gpt4o_predictions_comparison.csv", index=False)

print("Saved predictions and comparison to 'gpt4o_predictions_comparison.csv'")


In [ ]:
# o3 save

In [15]:
# Extract the first character of the GPT-4o prediction (e.g., 'A' from 'A: ...')
test_df_subset["gpto3_letter"] = test_df_subset["gpto3_direct_prediction"].astype(str).str.strip().str[0]

# Extract the correct answer letter from the ground truth label list
test_df_subset["answer_letter"] = test_df_subset["label"].apply(
    lambda x: str(x[0]).strip().upper() if isinstance(x, list) and len(x) > 0 else None
)

# Compare GPT-4o prediction to the ground truth letter
test_df_subset["gpto3_letter_match"] = test_df_subset.apply(
    lambda row: "Correct" if row["gpto3_letter"] == row["answer_letter"] else "Incorrect",
    axis=1
)

# Compute accuracy
correct_count = (test_df_subset["gpto3_letter_match"] == "Correct").sum()
total_count = test_df_subset["gpto3_letter_match"].notna().sum()
accuracy = correct_count / total_count if total_count > 0 else 0

# Print evaluation results
print(f"Letter-Based Correct Predictions: {correct_count}")
print(f"Total Predictions Compared: {total_count}")
print(f"Letter Match Accuracy: {accuracy:.2%}")


Letter-Based Correct Predictions: 688
Total Predictions Compared: 1861
Letter Match Accuracy: 36.97%


In [13]:
# Extract the predicted letter from GPT-4o's output (e.g., 'A' from 'A: ...')
test_df_subset["gpto3_letter"] = test_df_subset["gpto3_direct_prediction"].astype(str).str.strip().str[0]

# Extract the ground truth letter from the label list
test_df_subset["answer_letter"] = test_df_subset["label"].apply(
    lambda x: str(x[0]).strip().upper() if isinstance(x, list) and x else None
)

# Compare predicted letter with ground truth
test_df_subset["gpto3_letter_match"] = test_df_subset.apply(
    lambda row: "Correct" if row["gpto3_letter"] == row["gpto3_letter"] else "Incorrect",
    axis=1
)

# Compute accuracy
correct_count = (test_df_subset["gpto3_letter_match"] == "Correct").sum()
total_count = test_df_subset["gpto3_letter_match"].notna().sum()
accuracy = correct_count / total_count if total_count > 0 else 0

# Print results
print(f"Letter-Based Correct Predictions: {correct_count}")
print(f"Total Predictions Compared: {total_count}")
print(f"Letter Match Accuracy: {accuracy:.2%}")


Letter-Based Correct Predictions: 1861
Total Predictions Compared: 1861
Letter Match Accuracy: 100.00%


In [17]:
# Select relevant columns to save
columns_to_save = ["gpto3_direct_prediction", "label", "gpto3_letter", "answer_letter", "gpt_letter_match"]

# Save to CSV
test_df_subset[columns_to_save].to_csv("gpto3_bullets_comparison.csv", index=False)

print("Saved predictions and comparison to 'gpto3_bullets_comparison.csv'")


KeyError: "['gpt_letter_match'] not in index"